In [2]:
!pip install boto3

   ---------------------------------------- 0.0/16.0 MB ? eta -:--:--
   --- ------------------------------------ 1.6/16.0 MB 8.7 MB/s eta 0:00:02
   --------- ------------------------------ 3.9/16.0 MB 10.2 MB/s eta 0:00:02
   --------------- ------------------------ 6.3/16.0 MB 10.7 MB/s eta 0:00:01
   --------------------- ------------------ 8.7/16.0 MB 10.7 MB/s eta 0:00:01
   --------------------------- ------------ 11.0/16.0 MB 10.7 MB/s eta 0:00:01
   --------------------------------- ------ 13.4/16.0 MB 10.8 MB/s eta 0:00:01
   ---------------------------------------  15.7/16.0 MB 10.8 MB/s eta 0:00:01
   ---------------------------------------- 16.0/16.0 MB 10.5 MB/s  0:00:01

  Attempting uninstall: botocore

    Found existing installation: botocore 1.40.46

   ---------------------------------------- 0/3 [botocore]
   ---------------------------------------- 0/3 [botocore]
    Uninstalling botocore-1.40.46:
   ---------------------------------------- 0/3 [botocore]
      Su

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 2.25.0 requires botocore<1.40.50,>=1.40.46, but you have botocore 1.43.107 which is incompatible.


In [3]:
import pandas as pd
import boto3
from io import StringIO

s3 = boto3.client(
    "s3",
    aws_access_key_id="YCAJEgDwQoh8kx3zuEHaGYtKN",
    aws_secret_access_key="YCN1BIoOQn0y_HJMdtv7lQRxa1m3Wpi2Dqsga6B8",
    endpoint_url="https://storage.yandexcloud.net",
)
bucket = "s3-ds-20260915-7e2a105f57"

def read_s3_csv(key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(obj["Body"])

df_09 = read_s3_csv("run_date=2025-09-01/batch_features.csv")
df_10 = read_s3_csv("run_date=2025-10-01/batch_features.csv")

print(f"2025-09-01: {len(df_09)} строк, {len(df_09.columns)} колонок")
print(f"2025-10-01: {len(df_10)} строк, {len(df_10.columns)} колонок")

# Одинаковый набор колонок?
assert set(df_09.columns) == set(df_10.columns), "Колонки отличаются!"
print("✅ Набор полей одинаковый")

# Одинаковые customer_id?
assert set(df_09["customer_id"]) == set(df_10["customer_id"]), "customer_id отличаются!"
print("✅ customer_id совпадают")

# Разные даты срезов?
assert df_09["run_date"].iloc[0] != df_10["run_date"].iloc[0], "run_date одинаковые!"
print("✅ Даты срезов разные")

# Значения признаков отличаются?
numeric_cols = [c for c in df_09.columns if c not in ("customer_id", "run_date")]
diff_count = (df_09[numeric_cols].values != df_10[numeric_cols].values).sum()
print(f"Отличающихся значений признаков: {diff_count} из {df_09.shape[0] * len(numeric_cols)}")
assert diff_count > 0, "Признаки идентичны — что-то не так!"
print("✅ Значения признаков различаются")

2025-09-01: 17334 строк, 19 колонок
2025-10-01: 17590 строк, 19 колонок
✅ Набор полей одинаковый


AssertionError: customer_id отличаются!

In [4]:
ids_09 = set(df_09["customer_id"])
ids_10 = set(df_10["customer_id"])

only_in_09 = ids_09 - ids_10
only_in_10 = ids_10 - ids_09

print(f"Клиентов только в 2025-09-01: {len(only_in_09)}")
print(f"Клиентов только в 2025-10-01: {len(only_in_10)}")
print(f"Общих клиентов: {len(ids_09 & ids_10)}")

# Если хочешь посмотреть примеры — выведи первые 10 ID из каждой группы
if len(only_in_09) > 0:
    print("Примеры ID только в 09:", list(only_in_09)[:10])
if len(only_in_10) > 0:
    print("Примеры ID только в 10:", list(only_in_10)[:10])


Клиентов только в 2025-09-01: 20
Клиентов только в 2025-10-01: 276
Общих клиентов: 17314
Примеры ID только в 09: [19204, 9358, 11155, 18453, 8598, 3354, 18848, 7335, 3771, 9022]
Примеры ID только в 10: [2562, 13826, 13827, 1029, 19459, 10248, 2569, 4105, 17932, 1552]
